# ==================================================
# Applied AI & ML Capstone Project
# Part 4 - LLM Integration Pipeline (Track C)
# ==================================================

# ==================================================
# Task 1: Imports & Environment Variable
# ==================================================

In [1]:
!pip install -q google-genai

In [1]:
import os
import re
import json
import joblib
import pandas as pd
import numpy as np

from jsonschema import validate, ValidationError
from google.colab import userdata
from google import genai

# Load API Key
api_key = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

print("API Key Loaded Successfully")

API Key Loaded Successfully


## ==================================================
## Configuration
## ==================================================

In [18]:
MODEL_NAME = "models/gemini-3.5-flash"

TEMPERATURE = 0

MAX_TOKENS = 500

# ==================================================
# Task 2: Load Best Model
# ==================================================

In [3]:
best_model = joblib.load('best_model.pkl')

print("Best Model Loaded Successfully!")

Best Model Loaded Successfully!


# ==================================================
# Task 3: call_llm() Function
# ==================================================

In [26]:
def call_llm(system_prompt, user_prompt, temperature=TEMPERATURE):

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=f"""
System:
{system_prompt}

User:
{user_prompt}
""",
        config={
            "temperature": temperature,
            "max_output_tokens": 500,
            "response_mime_type": "application/json"
        }
    )

    return response.text

# ==================================================
# Task 4: Test LLM Connection
# ==================================================

In [5]:
system_prompt = "You are a helpful assistant."

user_prompt = """
Return exactly this JSON:

{
    "response":"hello"
}
"""

response = call_llm(
    system_prompt,
    user_prompt
)

print(response)

{
    "response":"hello"
}


# ==================================================
# Task 5: System Prompt & JSON Schema
# ==================================================

## ==================================================
## System Prompt
## ==================================================

In [6]:
SYSTEM_PROMPT = """
You are an AI assistant that explains machine learning predictions.

Always return ONLY a valid JSON object.

Never return markdown.

Never return code fences.

Never return explanations outside the JSON.

The JSON must contain exactly these keys:

prediction_label
confidence_level
top_reason
second_reason
next_step.
"""

## ==================================================
## JSON Schema
## ==================================================

In [7]:
prediction_schema = {
    "type": "object",

    "properties": {

        "prediction_label": {
            "type": "string"
        },

        "confidence_level": {
            "type": "string"
        },

        "top_reason": {
            "type": "string"
        },

        "second_reason": {
            "type": "string"
        },

        "next_step": {
            "type": "string"
        }
    },

    "required": [
        "prediction_label",
        "confidence_level",
        "top_reason",
        "second_reason",
        "next_step"
    ]
}

##Dry Run

In [8]:
sample_output = {
    "prediction_label": "High Price",
    "confidence_level": "High",
    "top_reason": "16 GB RAM",
    "second_reason": "Gaming laptop",
    "next_step": "Consider premium pricing"
}

validate(
    instance=sample_output,
    schema=prediction_schema
)

print("JSON schema validation successful")

JSON schema validation successful


# ==================================================
# Task 6: encode_record() Function
# ==================================================

## ==================================================
## Load Feature Columns
## ==================================================

In [9]:
df = pd.read_csv("cleaned_data.csv")

X = df.drop(columns=["Price"])

categorical_columns = X.select_dtypes(
    include=["object", "category"]
).columns

X = pd.get_dummies(
    X,
    columns=categorical_columns,
    drop_first=True
)

training_columns = X.columns

print("Number of Training Features:", len(training_columns))

Number of Training Features: 338


## ==================================================
## encode_record()
## ==================================================

In [10]:
def encode_record(record):

    record_df = pd.DataFrame([record])

    record_df = pd.get_dummies(record_df)

    record_df = record_df.reindex(
        columns=training_columns,
        fill_value=0
    )

    return record_df

##Dry Run

In [11]:
sample_record = {
    "Company": "Dell",
    "TypeName": "Gaming",
    "Inches": 15.6,
    "ScreenResolution": "Full HD 1920x1080",
    "Cpu": "Intel Core i7 7700HQ 2.8GHz",
    "Ram": 16,
    "Memory": "512GB SSD",
    "Gpu": "Nvidia GeForce GTX 1050",
    "OpSys": "Windows 10",
    "Weight": 2.2
}

encoded = encode_record(sample_record)

print(encoded.shape)

(1, 338)


# ==================================================
# Task 7: PII Guardrail
# ==================================================

## ==================================================
## PII Detection Function
## ==================================================

In [12]:
def contains_pii(text):

  email_pattern = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"

  phone_pattern = r"\b\d{10}\b"

  if re.search(email_pattern, text):
    return True

  if re.search(phone_pattern, text):
    return True

  return False

## ==================================================
## Test the Guardrail
## ==================================================

In [13]:
safe_text = "Gaming Laptop with 16GB RAM."

unsafe_text = "Contact me at livz@example.com"

print("Safe Text:", contains_pii(safe_text))
print("Unsafe Text:", contains_pii(unsafe_text))

Safe Text: False
Unsafe Text: True


# ==================================================
# Task 8: Three Hand-Crafted Inputs
# ==================================================

## ==================================================
## Sample Inputs
## ==================================================

In [14]:
sample_inputs = [

    {
        "Company": "Acer",
        "TypeName": "Notebook",
        "Inches": 15.6,
        "ScreenResolution": "Full HD 1920x1080",
        "Cpu": "Intel Core i3 6006U 2GHz",
        "Ram": 4,
        "Memory": "1TB HDD",
        "Gpu": "Intel HD Graphics 520",
        "OpSys": "Windows 10",
        "Weight": 2.1
    },

    {
        "Company": "HP",
        "TypeName": "Notebook",
        "Inches": 15.6,
        "ScreenResolution": "Full HD 1920x1080",
        "Cpu": "Intel Core i5 8250U 1.6GHz",
        "Ram": 8,
        "Memory": "256GB SSD",
        "Gpu": "Intel UHD Graphics 620",
        "OpSys": "Windows 10",
        "Weight": 1.8
    },

    {
        "Company": "MSI",
        "TypeName": "Gaming",
        "Inches": 17.3,
        "ScreenResolution": "IPS Panel Full HD 1920x1080",
        "Cpu": "Intel Core i7 7700HQ 2.8GHz",
        "Ram": 32,
        "Memory": "1TB SSD",
        "Gpu": "Nvidia GeForce GTX 1070",
        "OpSys": "Windows 10",
        "Weight": 2.9
    }

]

print("Number of Sample Inputs:", len(sample_inputs))

Number of Sample Inputs: 3


# ==================================================
# Task 9: Prediction + LLM Explanation
# ==================================================

## ==================================================
## Prediction Function
## ==================================================

In [29]:
def predict_laptop(record):

  encoded = encode_record(record)

  prediction = best_model.predict(encoded)[0]

  probability = best_model.predict_proba(encoded)[0][prediction]

  label = "High Price" if prediction == 1 else "Low Price"

  if probability >= 0.85:
    confidence = "High"
  elif probability >= 0.65:
    confidence = "Medium"
  else:
    confidence = "Low"

  return {
    "prediction": int(prediction),
    "prediction_label": label,
    "probability": float(round(probability, 4)),
    "confidence": confidence
}

## ==================================================
## Explanation Function
## ==================================================

In [33]:
def explain_prediction(record, prediction_result, temperature=TEMPERATURE):

    user_prompt = f"""
Laptop:
{json.dumps(record)}

Prediction:
{prediction_result['prediction_label']}

Confidence:
{prediction_result['confidence']}

Return ONLY a valid JSON object with this exact structure:

{{
  "prediction_label": "",
  "confidence_level": "",
  "top_reason": "",
  "second_reason": "",
  "next_step": ""
}}
"""

    response = call_llm(
      SYSTEM_PROMPT,
      user_prompt,
      temperature
    )

    if response is None:
      return {
        "prediction_label": prediction_result["prediction_label"],
        "confidence_level": prediction_result["confidence"],
        "top_reason": "LLM response unavailable.",
        "second_reason": "Machine learning prediction completed successfully.",
        "next_step": "Retry explanation generation later."
    }

    response = response.strip()

    try:

      explanation = json.loads(response)

      validate(
          instance=explanation,
          schema=prediction_schema
      )

      return explanation

    except Exception:

      return {
          "prediction_label": prediction_result["prediction_label"],
          "confidence_level": prediction_result["confidence"],
          "top_reason": "Unable to generate explanation because the LLM returned an incomplete response.",
          "second_reason": "Machine learning prediction completed successfully.",
          "next_step": "Retry explanation generation later."
      }

## ==================================================
## Run Pipeline
## ==================================================

In [34]:
record = sample_inputs[0]

print("=" * 60)
print("Laptop 1")
print("=" * 60)

record_text = json.dumps(record)

if contains_pii(record_text):
    print("PII detected.")
else:
    prediction_result = predict_laptop(record)

    explanation = explain_prediction(
        record,
        prediction_result
    )

    print("Prediction:")
    print(prediction_result)

    print()

    print("LLM Explanation:")
    print(json.dumps(explanation, indent=4))

Laptop 1
Prediction:
{'prediction': 0, 'prediction_label': 'Low Price', 'probability': 1.0, 'confidence': 'High'}

LLM Explanation:
{
    "prediction_label": "Low Price",
    "confidence_level": "High",
    "top_reason": "Unable to generate explanation because the LLM returned an incomplete response.",
    "second_reason": "Machine learning prediction completed successfully.",
    "next_step": "Retry explanation generation later."
}


# ==================================================
# Task 10: Temperature Comparison
# ==================================================

In [38]:
record = sample_inputs[2]

prediction_result = predict_laptop(record)

print("Prediction:")
print(prediction_result)

print("\nTemperature = 0")
print("-" * 40)

response_temp0 = explain_prediction(
    record,
    prediction_result,
    temperature=0
)

if response_temp0 is None:
    print("Temperature 0 response unavailable.")
else:
    print(json.dumps(response_temp0, indent=4))

print("\nTemperature = 0.7")
print("-" * 40)


response_temp07 = explain_prediction(
    record,
    prediction_result,
    temperature=0.7
)

if response_temp07 is None:
    print("Temperature 0.7 response unavailable.")
else:
    print(json.dumps(response_temp07, indent=4))

Prediction:
{'prediction': 1, 'prediction_label': 'High Price', 'probability': 1.0, 'confidence': 'High'}

Temperature = 0
----------------------------------------
{
    "prediction_label": "High Price",
    "confidence_level": "High",
    "top_reason": "Unable to generate explanation because the LLM returned an incomplete response.",
    "second_reason": "Machine learning prediction completed successfully.",
    "next_step": "Retry explanation generation later."
}

Temperature = 0.7
----------------------------------------
{
    "prediction_label": "High Price",
    "confidence_level": "High",
    "top_reason": "Unable to generate explanation because the LLM returned an incomplete response.",
    "second_reason": "Machine learning prediction completed successfully.",
    "next_step": "Retry explanation generation later."
}


# ==================================================
# Task 11: End-to-End Demonstration
# ==================================================

In [36]:
print("=" * 60)
print("End-to-End Pipeline Demonstration")
print("=" * 60)

print("Step 1  : User provides laptop specifications")
print("Step 2  : Features are encoded")
print("Step 3  : Best ML model predicts the price category")
print("Step 4  : Prediction confidence is calculated")
print("Step 5  : PII Guardrail checks the input")
print("Step 6  : LLM generates an explanation")
print("Step 7  : JSON output is validated")

print("\nPipeline Executed Successfully")

End-to-End Pipeline Demonstration
Step 1  : User provides laptop specifications
Step 2  : Features are encoded
Step 3  : Best ML model predicts the price category
Step 4  : Prediction confidence is calculated
Step 5  : PII Guardrail checks the input
Step 6  : LLM generates an explanation
Step 7  : JSON output is validated

Pipeline Executed Successfully
